# IDH Mutation Classification - No-Mask 3-Channel Pipeline (ALL slices)

ResNet-18 on contrast-agent-free brain MRI (**T2 + FLAIR + ASL**).

**Variant (no mask, all slices):** the three modalities are fed **as-is** - no tumor mask
and no mask channel, a plain 3-channel input. This is the baseline against which the
soft-mask variants are compared. It keeps **all axial slices** whose tumor area
>= `min_tumor_voxels` (= 250, with a top-K fallback) and follows the **all-slices recipe**
(lr 1.5e-5, dropout 0.3, 40 epochs).

Patient-level decision = **mean** of the per-slice probabilities (primary); confidence-weighted
and max aggregations are also reported as a sensitivity check.

Dataset: UCSF-PDGM v5, patient-level 60/20/20 split (seed 63).

In [ ]:
import os
os.environ["TORCH_HOME"]   = "/media/omeryasincur/Elements/omeryasin/torch_cache"   # ImageNet pretrained weights -> Elements
os.environ["MPLCONFIGDIR"] = "/media/omeryasincur/Elements/omeryasin/mpl_cache"      # matplotlib cache -> Elements
import re
import json
import random
import time
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from scipy.ndimage import zoom
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    accuracy_score, confusion_matrix, roc_curve,
)

warnings.filterwarnings("ignore")

In [ ]:
# --- Paths ---
BASE_DIR    = "/media/omeryasincur/Elements/PKG - UCSF-PDGM Version 5"
DATA_DIR    = os.path.join(BASE_DIR, "UCSF-PDGM-v5")
CSV_PATH    = os.path.join(BASE_DIR, "UCSF-PDGM-metadata_v5.csv")
SPLITS_DIR  = "/media/omeryasincur/Elements/omeryasin/idh_splits"
CACHE_DIR   = "/media/omeryasincur/Elements/omeryasin/cache_whole_brain_allslices_v250"   # on Elements drive (shared with soft-mask NB)
RESULTS_DIR = "/media/omeryasincur/Elements/omeryasin/pipeline_nomask3ch_allslices_results"  # on Elements
PLOTS_DIR   = os.path.join(RESULTS_DIR, "plots")
for d in [SPLITS_DIR, CACHE_DIR, RESULTS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)

# --- Config (matches the P1-all / P2-all all-slices recipe) ---
CFG = {
    "seed":                 63,
    # Data
    "test_fraction":        0.20,        # 99 patients
    "val_fraction":         0.20,        # 99 patients (of total)
    "min_tumor_voxels":     250,         # a slice is kept if tumor area >= this
    "min_slices_fallback":  5,           # if fewer than this qualify, fall back to top-K by area
    "min_tumor_pixels":     50,
    "clip_percentile":      99.5,
    "brain_bbox_margin":    0.02,
    "modalities":           ["T2", "FLAIR", "ASL"],   # plain 3-channel input (no mask)  (keys; shown as T2w/FLAIR/ASL in figures & report)
    "expected_patients":    495,
    # Model input
    "img_size":             224,
    # Training
    "batch_size":           16,
    "num_workers":          4,
    "max_epochs":           40,
    "patience":             999,         # early stopping disabled -> train full 40 epochs
    "use_weighted_sampler": True,        # class balance via sampling, not loss weighting
    "lr":                   3e-5,
    "weight_decay":         1e-4,
    "warmup_epochs":        10,          # long, slow warmup (0.001 -> 1.0)
    # Model
    "pretrained":           True,
    "dropout":              0.2,
    # Augmentation
    "aug_hflip":            True,
    "aug_vflip":            True,
    "aug_rotation":         20,
    "aug_scale":            (0.85, 1.15),
    "aug_gamma":            (0.8, 1.2),
    "aug_noise_std":        0.02,
    # Label smoothing
    "label_smoothing":      0.1,
    # Differential learning-rate multipliers (NO layers frozen)
    "lr_mult_early":        0.1,
    "lr_mult_layer4":       1.0,
    "lr_mult_head":         2.0,
    # Cache
    "force_rebuild_cache":  False,
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name()}")

In [ ]:
BASELINE_PATTERN = re.compile(r'^UCSF-PDGM-\d+$')


def normalize_id(s):
    """Canonical 4-digit padded ID."""
    base = s.replace("_nifti", "")
    return f"UCSF-PDGM-{int(base.split('-')[-1]):04d}"


def is_baseline(s):
    """True if ID/folder is a baseline (preoperative) scan, not a follow-up."""
    return "_FU" not in s and bool(BASELINE_PATTERN.match(s.replace("_nifti", "")))


def load_nifti(path):
    return nib.load(path).get_fdata().astype(np.float32)


def get_brain_bbox(brain_mask_3d, margin=0.02):
    projection = brain_mask_3d.max(axis=2) > 0
    if projection.sum() == 0:
        H, W = brain_mask_3d.shape[:2]
        return 0, H - 1, 0, W - 1
    rows = np.any(projection, axis=1)
    cols = np.any(projection, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    H, W = brain_mask_3d.shape[:2]
    pad_r = int((rmax - rmin) * margin)
    pad_c = int((cmax - cmin) * margin)
    return (max(0, rmin - pad_r), min(H - 1, rmax + pad_r),
            max(0, cmin - pad_c), min(W - 1, cmax + pad_c))


def normalize_volume(volume, brain_mask, clip_pct=99.5):
    """clip -> z-score -> min-max [0,1], background = 0."""
    out = np.zeros_like(volume, dtype=np.float32)
    brain_vox = volume[brain_mask > 0]
    if len(brain_vox) == 0 or brain_vox.std() < 1e-8:
        return out
    lo = np.percentile(brain_vox, 100 - clip_pct)
    hi = np.percentile(brain_vox, clip_pct)
    clipped = np.clip(volume, lo, hi)
    brain_clipped = clipped[brain_mask > 0]
    mu, sigma = brain_clipped.mean(), brain_clipped.std()
    if sigma < 1e-8:
        return out
    z = (clipped - mu) / sigma
    z_brain = z[brain_mask > 0]
    zmin, zmax = z_brain.min(), z_brain.max()
    if zmax - zmin < 1e-8:
        return out
    normalized = (z - zmin) / (zmax - zmin)
    normalized[brain_mask == 0] = 0.0
    return normalized.astype(np.float32)


def resize_slice(img, target_size, order=1):
    h, w = img.shape
    if h == target_size and w == target_size:
        return img
    return zoom(img, (target_size / h, target_size / w), order=order)


def select_top_k_slices(seg_volume, k=10, min_pixels=50):
    areas = []
    for i in range(seg_volume.shape[2]):
        area = int((seg_volume[:, :, i] > 0).sum())
        if area >= min_pixels:
            areas.append((i, area))
    areas.sort(key=lambda x: x[1], reverse=True)
    return areas[:k]


def select_all_slices(seg_volume, min_voxels=100, min_slices=5):
    """All axial slices with tumor area >= min_voxels.
    Fallback: if fewer than min_slices qualify, take the top-min_slices by area
    (so every patient keeps at least some slices). Returns (slice_idx, area) tuples."""
    areas_all = [(i, int((seg_volume[:, :, i] > 0).sum())) for i in range(seg_volume.shape[2])]
    qualifying = [(i, a) for i, a in areas_all if a >= min_voxels]
    if len(qualifying) >= min_slices:
        qualifying.sort(key=lambda x: x[1], reverse=True)
        return qualifying
    areas_all.sort(key=lambda x: x[1], reverse=True)
    return areas_all[:min_slices]


# Self-test
assert normalize_id("UCSF-PDGM-004")        == "UCSF-PDGM-0004"
assert normalize_id("UCSF-PDGM-0004_nifti") == "UCSF-PDGM-0004"
assert is_baseline("UCSF-PDGM-004")          is True
assert is_baseline("UCSF-PDGM-0429_FU003d")  is False
print("Helper functions verified")

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Raw CSV rows: {len(df)}")

# IDH value distribution
print("\nIDH value distribution:")
print(df["IDH"].value_counts())

# Drop follow-up scans
mask_baseline = df["ID"].apply(is_baseline)
followup_ids  = df.loc[~mask_baseline, "ID"].tolist()
print(f"\nFollow-up scans dropped: {len(followup_ids)}")

df = df[mask_baseline].copy().reset_index(drop=True)
df["patient_id"] = df["ID"].apply(normalize_id)

# --- LABEL ENCODING: Mut = 1, WT = 0 ---
df["idh_label"] = (df["IDH"] != "wildtype").astype(int)

csv_pids    = set(df["patient_id"])
folder_pids = {
    normalize_id(p) for p in os.listdir(DATA_DIR)
    if p.startswith("UCSF-PDGM-") and is_baseline(p)
}
overlap = csv_pids & folder_pids

print(f"\nCSV baseline patients:    {len(csv_pids)}")
print(f"Folder baseline patients: {len(folder_pids)}")
print(f"Matching:                 {len(overlap)}")
assert len(overlap) == CFG["expected_patients"], f"Expected {CFG['expected_patients']}, got {len(overlap)}"

print("\nClass distribution (Mut=1, WT=0):")
print(f"  Mut (label=1): {(df['idh_label']==1).sum()}")
print(f"  WT  (label=0): {(df['idh_label']==0).sum()}")
print("\nSanity check passed")

In [ ]:
patient_df = df[["patient_id", "idh_label"]].copy().reset_index(drop=True)

# First split: 80/20 train+val vs test
trainval_df, test_df = train_test_split(
    patient_df,
    test_size=CFG["test_fraction"],
    stratify=patient_df["idh_label"],
    random_state=CFG["seed"],
)

# Second split: from train+val, take val (99/396 ~ 0.25 of the remaining)
val_size_of_remaining = CFG["val_fraction"] / (1.0 - CFG["test_fraction"])
train_df_p, val_df_p = train_test_split(
    trainval_df,
    test_size=val_size_of_remaining,
    stratify=trainval_df["idh_label"],
    random_state=CFG["seed"],
)

train_pids = train_df_p["patient_id"].tolist()
val_pids   = val_df_p["patient_id"].tolist()
test_pids  = test_df["patient_id"].tolist()

# Verify no overlap
assert len(set(train_pids) & set(val_pids)) == 0
assert len(set(train_pids) & set(test_pids)) == 0
assert len(set(val_pids) & set(test_pids)) == 0
assert len(train_pids) + len(val_pids) + len(test_pids) == CFG["expected_patients"]

split_info = {"seed": CFG["seed"], "train": sorted(train_pids),
              "val": sorted(val_pids), "test": sorted(test_pids)}
with open(os.path.join(SPLITS_DIR, "split.json"), "w") as f:
    json.dump(split_info, f, indent=2)


def class_dist(pids):
    sub = patient_df[patient_df["patient_id"].isin(pids)]
    return f"Mut={int((sub['idh_label']==1).sum())}, WT={int((sub['idh_label']==0).sum())}"


print(f"Train: {len(train_pids)} patients ({class_dist(train_pids)})")
print(f"Val:   {len(val_pids)} patients ({class_dist(val_pids)})")
print(f"Test:  {len(test_pids)} patients ({class_dist(test_pids)})")
print(f"\nSplit saved to: {os.path.join(SPLITS_DIR, 'split.json')}")

In [ ]:
manifest_path = os.path.join(CACHE_DIR, "manifest.csv")
label_map = dict(zip(df["patient_id"], df["idh_label"]))

if os.path.exists(manifest_path) and not CFG["force_rebuild_cache"]:
    print(f"Cache exists at {CACHE_DIR}, skipping build.")
else:
    print(f"Building cache at {CACHE_DIR} ...")
    manifest_rows = []
    skipped = []

    for pid_folder in tqdm(sorted(os.listdir(DATA_DIR))):
        if not pid_folder.startswith("UCSF-PDGM-"):
            continue
        base = pid_folder.replace("_nifti", "")
        if "_FU" in base:
            skipped.append((pid_folder, "follow_up_scan"))
            continue
        pid = normalize_id(base)
        if pid not in label_map:
            skipped.append((pid, "no_label"))
            continue

        folder = os.path.join(DATA_DIR, pid_folder)
        try:
            T2_raw    = load_nifti(os.path.join(folder, f"{base}_T2_bias.nii.gz"))
            FLAIR_raw = load_nifti(os.path.join(folder, f"{base}_FLAIR_bias.nii.gz"))
            ASL_raw   = load_nifti(os.path.join(folder, f"{base}_ASL.nii.gz"))
            tumor_seg = load_nifti(os.path.join(folder, f"{base}_tumor_segmentation.nii.gz"))
            brain_seg = load_nifti(os.path.join(folder, f"{base}_brain_segmentation.nii.gz"))
        except Exception as e:
            skipped.append((pid, f"load_error: {e}"))
            continue

        brain_mask = (brain_seg > 0).astype(np.uint8)
        tumor_mask = (tumor_seg > 0).astype(np.uint8)
        rmin, rmax, cmin, cmax = get_brain_bbox(brain_mask, margin=CFG["brain_bbox_margin"])

        T2_n    = normalize_volume(T2_raw,    brain_mask, clip_pct=CFG["clip_percentile"])
        FLAIR_n = normalize_volume(FLAIR_raw, brain_mask, clip_pct=CFG["clip_percentile"])
        ASL_n   = normalize_volume(ASL_raw,   brain_mask, clip_pct=CFG["clip_percentile"])

        top_slices = select_all_slices(tumor_seg, min_voxels=CFG["min_tumor_voxels"],
                                       min_slices=CFG["min_slices_fallback"])
        if not top_slices:
            skipped.append((pid, "no_tumor_slice"))
            continue

        patient_cache_dir = os.path.join(CACHE_DIR, pid)
        os.makedirs(patient_cache_dir, exist_ok=True)

        for rank, (k, area) in enumerate(top_slices):
            slice_path = os.path.join(patient_cache_dir, f"slice_{rank:02d}_k{k:03d}.npz")
            np.savez_compressed(
                slice_path,
                T2    = T2_n[rmin:rmax+1, cmin:cmax+1, k].astype(np.float16),
                FLAIR = FLAIR_n[rmin:rmax+1, cmin:cmax+1, k].astype(np.float16),
                ASL   = ASL_n[rmin:rmax+1, cmin:cmax+1, k].astype(np.float16),
                mask  = tumor_mask[rmin:rmax+1, cmin:cmax+1, k].astype(np.uint8),
                brain = brain_mask[rmin:rmax+1, cmin:cmax+1, k].astype(np.uint8),
            )
            manifest_rows.append({
                "patient_id": pid,
                "slice_path": slice_path,
                "k_axial":    int(k),
                "rank":       int(rank),
                "tumor_area": int(area),
                "idh_label":  int(label_map[pid]),
            })

    pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)
    print(f"\nCache build complete: {len(manifest_rows)} slices")

manifest = pd.read_csv(manifest_path)
manifest["idh_label"] = manifest["patient_id"].map(label_map)
manifest.to_csv(manifest_path, index=False)
print(f"\nLoaded manifest: {len(manifest)} slices, {manifest['patient_id'].nunique()} patients")
print(f"Slice-level: Mut={int((manifest['idh_label']==1).sum())}, "
      f"WT={int((manifest['idh_label']==0).sum())}")

In [ ]:
# --- Preview: the 3-channel input (no mask) ---
rng = np.random.default_rng(CFG["seed"])
mut_pid = rng.choice(manifest[manifest["idh_label"] == 1]["patient_id"].unique())

rows = manifest[manifest["patient_id"] == mut_pid]
row  = rows.iloc[len(rows) // 2]
data = np.load(row["slice_path"])
ch = {"T2": data["T2"], "FLAIR": data["FLAIR"], "ASL": data["ASL"]}

fig, axes = plt.subplots(1, 4, figsize=(18, 5)); fig.patch.set_facecolor("white")
for ax, name in zip(axes[:3], ["T2", "FLAIR", "ASL"]):
    ax.imshow(np.rot90(ch[name]), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{name}", fontsize=13, fontweight="bold"); ax.axis("off")
rgb = np.stack([np.rot90(ch["T2"]), np.rot90(ch["FLAIR"]), np.rot90(ch["ASL"])], axis=-1)
axes[3].imshow(np.clip(rgb, 0, 1))
axes[3].set_title("T2/FLAIR/ASL (RGB)", fontsize=13, fontweight="bold"); axes[3].axis("off")
fig.suptitle(f"{mut_pid}  |  k={row['k_axial']}  |  tumor area={row['tumor_area']}  |  no mask (3-channel)",
             fontsize=14, fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
class IDHSliceDataset(Dataset):
    """Plain 3-channel dataset (NO mask): T2/FLAIR/ASL fed as-is, with no tumor mask
    and no mask channel. This is the baseline for the soft-mask comparison."""

    def __init__(self, manifest_df, cfg, augment=False):
        self.df       = manifest_df.reset_index(drop=True)
        self.cfg      = cfg
        self.augment  = augment
        self.img_size = cfg["img_size"]
        self.mods     = cfg["modalities"]            # ["T2", "FLAIR", "ASL"]

    def __len__(self):
        return len(self.df)

    def _augment(self, img):
        if self.cfg["aug_hflip"] and random.random() < 0.5:
            img = TF.hflip(img)
        if self.cfg.get("aug_vflip", False) and random.random() < 0.5:
            img = TF.vflip(img)
        if self.cfg["aug_rotation"] > 0:
            angle = random.uniform(-self.cfg["aug_rotation"], self.cfg["aug_rotation"])
            img = TF.rotate(img, angle, interpolation=TF.InterpolationMode.BILINEAR)
        if self.cfg["aug_scale"] is not None:
            sc = random.uniform(*self.cfg["aug_scale"])
            img = TF.affine(img, angle=0, translate=[0, 0], scale=sc, shear=0,
                            interpolation=TF.InterpolationMode.BILINEAR)
        if random.random() < 0.5:
            gamma = random.uniform(*self.cfg["aug_gamma"])
            img = torch.clamp(img, 0, 1) ** gamma
        if random.random() < 0.3 and self.cfg["aug_noise_std"] > 0:
            img = img + torch.randn_like(img) * self.cfg["aug_noise_std"]
            img = torch.clamp(img, 0, 1)
        return img

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        data = np.load(row["slice_path"])
        # No mask: plain 3-channel input (T2/FLAIR/ASL as-is).
        modality_resized = [
            resize_slice(data[m].astype(np.float32), self.img_size, order=1)
            for m in self.mods
        ]
        img = np.stack(modality_resized, axis=0).astype(np.float32)
        img_tensor = torch.from_numpy(img)
        if self.augment:
            img_tensor = self._augment(img_tensor)
        return {
            "image":      img_tensor,
            "label":      torch.tensor(row["idh_label"], dtype=torch.float32),
            "patient_id": row["patient_id"],
        }


# Sanity check
_test_ds = IDHSliceDataset(manifest, CFG, augment=True)
_s = _test_ds[0]
print(f"Dataset size: {len(_test_ds)}")
print(f"Sample 0: image shape={tuple(_s['image'].shape)}, "
      f"range=[{_s['image'].min():.3f}, {_s['image'].max():.3f}], "
      f"label={_s['label'].item()}, pid={_s['patient_id']}")
assert _s["image"].shape == (3, CFG["img_size"], CFG["img_size"])

In [ ]:
def build_model(pretrained=True, dropout=0.5):
    weights = torchvision.models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = torchvision.models.resnet18(weights=weights)
    model.fc = nn.Sequential(
        nn.Dropout(dropout),
        nn.Linear(512, 1),
    )
    return model


_m = build_model(pretrained=CFG["pretrained"], dropout=CFG["dropout"]).to(DEVICE)
total = sum(p.numel() for p in _m.parameters())
print(f"Model: ResNet-18 (3-channel input, binary head, dropout={CFG['dropout']})")
print(f"  Parameters: {total:,}")
_m.eval()
with torch.no_grad():
    _x = torch.randn(2, 3, CFG["img_size"], CFG["img_size"]).to(DEVICE)
    _y = _m(_x)
print(f"  Forward: {tuple(_x.shape)} -> {tuple(_y.shape)}")
assert _y.shape == (2, 1)
del _m

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, label_smoothing=0.0):
    model.train()
    total_loss, n = 0.0, 0
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        if label_smoothing > 0:
            labels = labels * (1.0 - 2.0 * label_smoothing) + label_smoothing
        optimizer.zero_grad(set_to_none=True)
        logits = model(images).squeeze(-1)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        bs = labels.size(0)
        total_loss += loss.item() * bs
        n += bs
    return total_loss / n


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss, n = 0.0, 0
    all_logits, all_labels, all_pids = [], [], []
    for batch in loader:
        images = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        logits = model(images).squeeze(-1)
        loss = criterion(logits, labels)
        bs = labels.size(0)
        total_loss += loss.item() * bs
        n += bs
        all_logits.append(logits.cpu())
        all_labels.append(labels.cpu())
        all_pids.extend(batch["patient_id"])
    return {
        "loss":   total_loss / n,
        "logits": torch.cat(all_logits),
        "labels": torch.cat(all_labels),
        "pids":   all_pids,
    }


print("Train/validate functions defined")

In [ ]:
def aggregate_to_patient(pids, slice_probs, slice_labels):
    """Mean probability across slices for each patient."""
    data = defaultdict(lambda: {"probs": [], "label": None})
    for pid, p, l in zip(pids, slice_probs, slice_labels):
        data[pid]["probs"].append(float(p))
        data[pid]["label"] = int(l)
    pids_out, probs_out, labels_out = [], [], []
    for pid, d in data.items():
        pids_out.append(pid)
        probs_out.append(np.mean(d["probs"]))
        labels_out.append(d["label"])
    return np.array(pids_out), np.array(probs_out), np.array(labels_out)


def find_f1_threshold(labels, probs):
    """Threshold that maximizes F1 (balanced precision/recall). Selected on val."""
    if len(np.unique(labels)) < 2:
        return 0.5
    best_f1, best_thr = -1.0, 0.5
    candidate_thrs = sorted(set(np.round(probs, 4).tolist() + [0.5]))
    for thr in candidate_thrs:
        preds = (probs >= thr).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return float(best_thr)


def find_youden_threshold(labels, probs):
    """Kept for reference; F1 is primary."""
    if len(np.unique(labels)) < 2:
        return 0.5
    fpr, tpr, thresholds = roc_curve(labels, probs)
    return float(thresholds[np.argmax(tpr - fpr)])


def compute_metrics(labels, probs, threshold):
    """Mut=1 convention: Recall = sensitivity for Mut, Specificity = correct WT."""
    labels = np.asarray(labels).astype(int)
    preds  = (probs >= threshold).astype(int)
    try:
        auc = roc_auc_score(labels, probs) if len(np.unique(labels)) >= 2 else float("nan")
    except ValueError:
        auc = float("nan")
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        "AUC":         auc,
        "Accuracy":    accuracy_score(labels, preds),
        "Precision":   precision_score(labels, preds, zero_division=0),
        "Recall":      recall_score(labels, preds, zero_division=0),
        "F1":          f1_score(labels, preds, zero_division=0),
        "Specificity": tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        "Threshold":   threshold,
        "TP": int(tp), "TN": int(tn), "FP": int(fp), "FN": int(fn),
    }


print("Metrics + thresholds defined (F1-optimal = primary)")

In [ ]:
# --- Class balancing via WeightedRandomSampler (NOT pos_weight) ---
# Each batch is ~50/50 Mut/WT in expectation.
n_wt  = (manifest["idh_label"] == 0).sum()
n_mut = (manifest["idh_label"] == 1).sum()
print(f"Class balance (slice-level): Mut={n_mut}, WT={n_wt}")
print("Using WeightedRandomSampler for balanced batches (unweighted BCE)")

set_seed(CFG["seed"])

train_manifest = manifest[manifest["patient_id"].isin(train_pids)].copy().reset_index(drop=True)
val_manifest   = manifest[manifest["patient_id"].isin(val_pids)].copy().reset_index(drop=True)
test_manifest  = manifest[manifest["patient_id"].isin(test_pids)].copy().reset_index(drop=True)

train_ds = IDHSliceDataset(train_manifest, CFG, augment=True)
val_ds   = IDHSliceDataset(val_manifest,   CFG, augment=False)
test_ds  = IDHSliceDataset(test_manifest,  CFG, augment=False)

# Per-slice weights for balanced sampling
train_class_counts = train_manifest["idh_label"].value_counts().to_dict()
train_weights = train_manifest["idh_label"].map(lambda l: 1.0 / train_class_counts[l]).values
train_sampler = torch.utils.data.WeightedRandomSampler(
    weights=torch.DoubleTensor(train_weights),
    num_samples=len(train_manifest),
    replacement=True,
)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], sampler=train_sampler,
                          num_workers=CFG["num_workers"], pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"], shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True)

print(f"\nTrain: {len(train_ds)} slices, {len(train_pids)} patients")
print(f"Val:   {len(val_ds)} slices, {len(val_pids)} patients")
print(f"Test:  {len(test_ds)} slices, {len(test_pids)} patients")

model = build_model(pretrained=CFG["pretrained"], dropout=CFG["dropout"]).to(DEVICE)

# Differential LR: early layers learn slowly, head fastest. No layers frozen.
early_params = (
    list(model.conv1.parameters()) + list(model.bn1.parameters()) +
    list(model.layer1.parameters()) + list(model.layer2.parameters()) +
    list(model.layer3.parameters())
)
optimizer = torch.optim.AdamW([
    {"params": early_params,              "lr": CFG["lr"] * CFG["lr_mult_early"]},
    {"params": model.layer4.parameters(), "lr": CFG["lr"] * CFG["lr_mult_layer4"]},
    {"params": model.fc.parameters(),     "lr": CFG["lr"] * CFG["lr_mult_head"]},
], weight_decay=CFG["weight_decay"])

# Warmup + cosine schedule
warmup_epochs = CFG.get("warmup_epochs", 5)
def lr_lambda(epoch):
    if epoch < warmup_epochs:
        # Slow linear warmup: 0.001 -> 1.0 (model barely moves in the first 1-2 epochs)
        return 0.001 + (1.0 - 0.001) * (epoch / warmup_epochs)
    progress = (epoch - warmup_epochs) / max(1, CFG["max_epochs"] - warmup_epochs)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

# Unweighted BCE - class balance handled by the sampler
criterion = nn.BCEWithLogitsLoss()

history = {"epoch": [], "train_loss": [], "val_loss": [], "val_auc": []}
best_val_auc = -1.0
best_ckpt_path = os.path.join(RESULTS_DIR, "best_model.pt")

t0 = time.time()
for epoch in range(1, CFG["max_epochs"] + 1):
    t0_ep = time.time()
    train_loss = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE,
        label_smoothing=CFG.get("label_smoothing", 0.0),
    )
    val_out = validate(model, val_loader, criterion, DEVICE)
    val_loss = float(val_out["loss"])
    val_probs  = torch.sigmoid(val_out["logits"]).numpy()
    val_labels = val_out["labels"].numpy().astype(int)
    val_auc = roc_auc_score(val_labels, val_probs) if len(np.unique(val_labels)) >= 2 else 0.5
    scheduler.step()

    history["epoch"].append(epoch)
    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(val_loss)
    history["val_auc"].append(float(val_auc))

    ckpt_improved = val_auc > best_val_auc + 1e-4
    if ckpt_improved:
        best_val_auc = val_auc
        torch.save(model.state_dict(), best_ckpt_path)

    ck = " *ckpt" if ckpt_improved else ""
    print(f"  Ep {epoch:02d}/{CFG['max_epochs']} | train_loss={train_loss:.4f} | "
          f"val_loss={val_loss:.4f} | val_AUC={val_auc:.4f}{ck} | {time.time()-t0_ep:.1f}s")
    # NOTE: no early stopping - fixed 40-epoch training

print(f"\nTotal training time: {(time.time()-t0)/60:.1f} min")
print(f"Best val_AUC: {best_val_auc:.4f}")

with open(os.path.join(RESULTS_DIR, "history.json"), "w") as f:
    json.dump(history, f, indent=2)

In [ ]:
model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE))

# --- Val predictions ---
val_out = validate(model, val_loader, criterion, DEVICE)
val_slice_probs  = torch.sigmoid(val_out["logits"]).numpy()
val_slice_labels = val_out["labels"].numpy().astype(int)
val_pat_pids, val_pat_probs, val_pat_labels = aggregate_to_patient(
    val_out["pids"], val_slice_probs, val_slice_labels
)

# --- Test predictions ---
test_out = validate(model, test_loader, criterion, DEVICE)
test_slice_probs  = torch.sigmoid(test_out["logits"]).numpy()
test_slice_labels = test_out["labels"].numpy().astype(int)
test_pat_pids, test_pat_probs, test_pat_labels = aggregate_to_patient(
    test_out["pids"], test_slice_probs, test_slice_labels
)

# --- Threshold strategies (all selected on val) ---
thr_f1     = find_f1_threshold(val_pat_labels, val_pat_probs)
thr_youden = find_youden_threshold(val_pat_labels, val_pat_probs)
thr_prev   = float((val_pat_labels == 1).sum() / len(val_pat_labels))

print(f"\n{'='*70}")
print("  No-Mask 3-Channel (all slices) - Threshold Comparison")
print(f"{'='*70}\n")

strategies = [("F1-optimal", thr_f1), ("Youden-J", thr_youden), ("Prevalence", thr_prev)]
results = {}
for name, thr in strategies:
    print(f"\n--- {name} (threshold = {thr:.4f}) ---")
    vm = compute_metrics(val_pat_labels,  val_pat_probs,  thr)
    tm = compute_metrics(test_pat_labels, test_pat_probs, thr)
    print(f"  {'Metric':<14} {'Val':>10}  {'Test':>10}")
    for k in ["AUC", "Accuracy", "Precision", "Recall", "F1", "Specificity"]:
        print(f"  {k:<14} {vm[k]:>10.4f}  {tm[k]:>10.4f}")
    print(f"  Test confusion: TP={tm['TP']:3d} TN={tm['TN']:3d} FP={tm['FP']:3d} FN={tm['FN']:3d}")
    results[name] = {"threshold": thr, "val": vm, "test": tm}

# --- Save: F1-optimal as the primary operating point ---
f1_threshold = thr_f1
val_metrics  = results["F1-optimal"]["val"]
test_metrics = results["F1-optimal"]["test"]

with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump({
        "threshold_f1_val":     thr_f1,
        "threshold_youden_val": thr_youden,
        "threshold_prevalence": thr_prev,
        "results": {name: {"threshold": r["threshold"], "val": r["val"], "test": r["test"]}
                    for name, r in results.items()},
        "primary_strategy": "F1-optimal",
    }, f, indent=2)

pd.DataFrame({
    "patient_id": val_pat_pids,  "true_label": val_pat_labels,
    "pred_prob":  val_pat_probs, "pred_label": (val_pat_probs >= f1_threshold).astype(int),
}).to_csv(os.path.join(RESULTS_DIR, "val_predictions.csv"), index=False)

pd.DataFrame({
    "patient_id": test_pat_pids, "true_label": test_pat_labels,
    "pred_prob":  test_pat_probs, "pred_label": (test_pat_probs >= f1_threshold).astype(int),
}).to_csv(os.path.join(RESULTS_DIR, "test_predictions.csv"), index=False)

print(f"\nSaved predictions at threshold (F1-optimal) = {f1_threshold:.4f}")

In [ ]:
# --- Plots: training dynamics + test confusion matrix ---
ep = history["epoch"]
fig, axes = plt.subplots(1, 3, figsize=(20, 5)); fig.patch.set_facecolor("white")

axes[0].plot(ep, history["train_loss"], label="train loss")
axes[0].plot(ep, history["val_loss"],   label="val loss")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("BCE loss")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(ep, history["val_auc"], color="tab:green")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("AUC")
axes[1].set_title("Validation AUC (slice-level)"); axes[1].set_ylim(0.5, 1.0)

cm = np.array([[test_metrics["TN"], test_metrics["FP"]],
               [test_metrics["FN"], test_metrics["TP"]]])
axes[2].imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm):
    axes[2].text(j, i, str(v), ha="center", va="center", fontsize=16, fontweight="bold")
axes[2].set_xticks([0, 1]); axes[2].set_xticklabels(["Pred WT", "Pred Mut"])
axes[2].set_yticks([0, 1]); axes[2].set_yticklabels(["True WT", "True Mut"])
axes[2].set_title(f"Test confusion (thr={f1_threshold:.3f})")

fig.suptitle(f"No-Mask 3-Channel (all slices)  |  Test AUC={test_metrics['AUC']:.3f}  "
             f"F1={test_metrics['F1']:.3f}  Acc={test_metrics['Accuracy']:.3f}",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "summary.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# --- Aggregation sensitivity: mean (PRIMARY) vs confidence-weighted vs max ---
# Post-hoc only (no retraining): same per-slice probabilities, different patient pooling.
from collections import defaultdict

def aggregate_by_mode(pids, probs, labels, mode="mean", gamma=1.0):
    d = defaultdict(lambda: {"p": [], "l": None})
    for pid, p, l in zip(pids, probs, labels):
        d[pid]["p"].append(float(p)); d[pid]["l"] = int(l)
    out_p, out_l = [], []
    for pid, v in d.items():
        p = np.asarray(v["p"])
        if mode == "mean":
            agg = float(p.mean())
        elif mode == "max":
            agg = float(p.max())
        elif mode == "weighted":
            eps = 1e-6; pc = np.clip(p, eps, 1 - eps)
            w = np.abs(np.log(pc / (1 - pc))) ** gamma     # confidence = |logit|, reference = 0.5
            agg = float((w * p).sum() / (w.sum() + 1e-8))
        out_p.append(agg); out_l.append(v["l"])
    return np.asarray(out_p), np.asarray(out_l)

print(f"\n{'='*80}")
print("  Aggregation sensitivity (patient-level)  -  MEAN is the primary decision rule")
print(f"{'='*80}")
print(f"  {'Aggregation':<22}{'thr':>7}{'AUC':>8}{'F1':>8}{'Prec':>8}{'Rec':>8}{'Spec':>8}{'Acc':>8}   confusion")
agg_results = {}
for mode, tag in [("mean", "mean (primary)"), ("weighted", "confidence-weighted"), ("max", "max")]:
    vp, vl = aggregate_by_mode(val_out["pids"],  val_slice_probs,  val_slice_labels, mode=mode)
    tp, tl = aggregate_by_mode(test_out["pids"], test_slice_probs, test_slice_labels, mode=mode)
    thr = find_f1_threshold(vl, vp)
    tm  = compute_metrics(tl, tp, thr)
    print(f"  {tag:<22}{thr:>7.3f}{tm['AUC']:>8.3f}{tm['F1']:>8.3f}{tm['Precision']:>8.3f}"
          f"{tm['Recall']:>8.3f}{tm['Specificity']:>8.3f}{tm['Accuracy']:>8.3f}   "
          f"TP{tm['TP']} TN{tm['TN']} FP{tm['FP']} FN{tm['FN']}")
    agg_results[mode] = {"threshold": thr, "test": tm}

with open(os.path.join(RESULTS_DIR, "aggregation_sensitivity.json"), "w") as f:
    json.dump(agg_results, f, indent=2)
print("\n  NOTE: 'mean' is the declared decision rule. 'weighted' (logit, gamma=1, ref 0.5) and 'max'")
print("        are reported only as a sensitivity check (single split, ~21 test mutants).")

In [ ]:
# --- GradCAM++ for binary classification (signed target) ---
class GradCAMpp:
    """target_class=1 -> regions supporting MUT; target_class=0 -> regions supporting WT."""
    def __init__(self, model, target_layer):
        self.model = model.eval(); self.target_layer = target_layer
        self.activations = None; self.gradients = None
        target_layer.register_forward_hook(self._fwd_hook)
        target_layer.register_full_backward_hook(self._bwd_hook)
    def _fwd_hook(self, m, i, o): self.activations = o.detach()
    def _bwd_hook(self, m, gi, go): self.gradients = go[0].detach()
    def __call__(self, x, target_class=1):
        x = x.requires_grad_(True)
        logits = self.model(x).squeeze(-1)
        self.model.zero_grad()
        sign = 1.0 if target_class == 1 else -1.0
        (sign * logits.sum()).backward()
        grads = self.gradients; acts = self.activations
        g2 = grads ** 2; g3 = grads ** 3
        denom = (acts * g3).sum(dim=[2, 3], keepdim=True)
        alpha = g2 / (2.0 * g2 + denom + 1e-8)
        weights = (alpha * F.relu(grads)).sum(dim=[2, 3], keepdim=True)
        cam = F.relu((weights * acts).sum(dim=1))
        cam = F.interpolate(cam.unsqueeze(1), size=x.shape[-2:], mode="bilinear", align_corners=False).squeeze(1)
        cmin = cam.amin(dim=[1, 2], keepdim=True); cmax = cam.amax(dim=[1, 2], keepdim=True)
        cam = (cam - cmin) / (cmax - cmin + 1e-8)
        return cam.detach().cpu().numpy(), torch.sigmoid(logits).detach().cpu().numpy()


def gradcam_grid(model, manifest_df, target_layer, device, n_per_class=8, ncols=4, cfg=CFG, seed=42):
    """Compact GradCAM++ over many patients: one panel each = heatmap over T2 + tumor contour.
    Input is the plain 3-channel (no mask) the model was trained on. Top rows = Mut, bottom = WT."""
    cam = GradCAMpp(model, target_layer); S = cfg["img_size"]
    r0 = manifest_df[manifest_df["rank"] == 0]
    idx1 = r0[r0["idh_label"] == 1].sample(n=min(n_per_class, int((r0["idh_label"] == 1).sum())), random_state=seed).index.tolist()
    idx0 = r0[r0["idh_label"] == 0].sample(n=min(n_per_class, int((r0["idh_label"] == 0).sum())), random_state=seed).index.tolist()
    indices = idx1 + idx0
    nrows = int(np.ceil(len(indices) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.1, nrows * 3.4)); fig.patch.set_facecolor("white")
    axes = np.asarray(axes).reshape(-1)
    for ax in axes: ax.axis("off")
    for ax, idx in zip(axes, indices):
        row = manifest_df.iloc[idx]; data = np.load(row["slice_path"])
        pid = row["patient_id"]; label = int(row["idh_label"])
        full  = [resize_slice(data[m].astype(np.float32), S, order=1) for m in ["T2", "FLAIR", "ASL"]]
        mask  = resize_slice(data["mask"].astype(np.float32),  S, order=0)
        brain = resize_slice(data["brain"].astype(np.float32), S, order=0)
        x = torch.from_numpy(np.stack(full, axis=0).astype(np.float32)).unsqueeze(0).to(device)
        hm, prob = cam(x, target_class=label)
        hm = hm[0] * brain; p = float(prob[0]); pred = int(p >= f1_threshold)
        ax.imshow(np.rot90(full[0]), cmap="gray", vmin=0, vmax=1)
        ax.imshow(np.rot90(hm), cmap="jet", alpha=0.5)
        ax.contour(np.rot90(mask), levels=[0.5], colors="white", linewidths=1.4)
        ok = "\u2713" if pred == label else "\u2717"
        tcol = "#1C7293" if label == 1 else "#B26A00"
        ax.set_title(f"{pid}\nTrue {'Mut' if label==1 else 'WT'} | Pred {'Mut' if pred==1 else 'WT'} ({p:.2f}) [{ok}]",
                     fontsize=9, fontweight="bold", color=tcol)
    fig.suptitle("GradCAM++ (no-mask model) - heatmap over T2 + tumor contour (white)   |   top = Mut, bottom = WT",
                 fontsize=14, fontweight="bold")
    plt.tight_layout(); plt.savefig(os.path.join(PLOTS_DIR, "gradcam_grid.png"), dpi=140, bbox_inches="tight"); plt.show()


model.load_state_dict(torch.load(best_ckpt_path, map_location=DEVICE)); model.eval()
test_manifest_reset = test_manifest.reset_index(drop=True)
gradcam_grid(model, test_manifest_reset, model.layer4, DEVICE, n_per_class=8, ncols=4)

In [ ]:
# ============================================================================
#  No-Mask 3ch  vs  Soft-Mask (alpha=0.5)  -  same split, same hyperparameters
#  Headline = AUC (threshold-free). Full panel at matched ~95% sensitivity.
#  Both thresholds are VALIDATION-selected (no-mask: val-prevalence;
#  soft-mask: val-F1, loaded from the grid notebook's saved summary).
#  Run this AFTER the evaluation cell (uses val_/test_ patient probs from it).
# ============================================================================
import json

# --- No-mask at validation-prevalence threshold (val-selected -> ~95% sensitivity) ---
nomask_thr = float((val_pat_labels == 1).sum() / len(val_pat_labels))
nomask_tm  = compute_metrics(test_pat_labels, test_pat_probs, nomask_thr)

# --- Soft-mask: grid-winner test metrics (val-F1 threshold), from saved summary ---
SOFT_SUMMARY = "/media/omeryasincur/Elements/omeryasin/pipeline_softmask3ch_grid/grid_final_summary.json"
try:
    soft_tm = json.load(open(SOFT_SUMMARY))["test_metrics"]
except (FileNotFoundError, KeyError):
    soft_tm = {"AUC": 0.987, "F1": 0.870, "Precision": 0.800, "Recall": 0.952,
               "Specificity": 0.936, "Accuracy": 0.939}
    print("(grid summary not found -> using recorded grid-winner numbers)")

K = ["AUC", "Recall", "Specificity", "Precision", "F1", "Accuracy"]
def _row(tag, tm):
    return f"  {tag:<18}" + "".join(f"{tm[k]:>9.3f}" for k in K)

print("=" * 74)
print("  No-Mask 3ch   vs   Soft-Mask (alpha=0.5)    |    same hyperparameters")
print("=" * 74)
print(f"\nHeadline (threshold-free):   AUC   no-mask = {nomask_tm['AUC']:.3f}   "
      f"soft-mask = {soft_tm['AUC']:.3f}   (delta = +{soft_tm['AUC'] - nomask_tm['AUC']:.3f})")
print(f"\nFull panel at matched sensitivity (~0.95):")
print("  " + " " * 18 + "".join(f"{h:>9}" for h in ["AUC", "Recall", "Spec", "Prec", "F1", "Acc"]))
print("  " + "-" * 72)
print(_row("no-mask 3ch", nomask_tm))
print(_row("soft-mask a=0.5", soft_tm))

cmp_df = pd.DataFrame(
    [{"model": "no-mask 3ch",     **{k: round(nomask_tm[k], 4) for k in K}},
     {"model": "soft-mask a=0.5", **{k: round(soft_tm[k], 4)   for k in K}}]
)
cmp_path = os.path.join(RESULTS_DIR, "nomask_vs_softmask_comparison.csv")
cmp_df.to_csv(cmp_path, index=False)
print(f"\nSaved: {cmp_path}")